# Develop and test a QEC protocol (qodec)

Use `qdk` and `qodec` to develop and test a quantum error correction protocol. Start with the C4 code and build a baseline qodec. Then inspect its gadgets and improve them. Along the way, use `qdk`'s built-in profile and audit features to debug and validate.

## Executive summary

```python
from qdk import ec
from qdk.simulation import run_qir

c4 = qodec.Code(stabilizers=..., x=..., z=...)  # define a code
protocol = ec.build_qodec(c4, strategy=...)  # build a qodec
ec.audit(protocol)  # validate the qodec
results = run_qir(qir_program, qodec=protocol, ...)  # simulate a logical program
```

Section 7 defines block-level benchmarks using the six C4 ISA instructions, with matched state-preparation-and-measurement (SPAM) references. The noisy sweep sends their named QIR calls through `run_qir` with the improved qodec, using 100 attempts per point. Runtime exceptions remain visible; no generic-gate or fault-table substitute is used.

## Install

```bash
pip install "qdk[ec]" matplotlib
```

## 1. Define your code

We will base our protocol on the $4$-qubit $[[4,2,2]]$ error-detecting code C4.
The C4 code stores two logical qubits in four physical qubits. Its two stabilizers are $XXXX$ and $ZZZZ$.

The `qodec` package is used to define the protocol; `qdk` provides the analysis.

In [38]:
import qodec

c4 = qodec.Code(
    "C4",
    stabilizers=["X_0 X_1 X_2 X_3", "Z_0 Z_1 Z_2 Z_3"],
    x=["X_0 X_1", "X_0 X_2"],
    z=["Z_0 Z_2", "Z_0 Z_1"],
)

Pauli strings in `qodec` use the sparse notation `P_i`, where `P` is one of the Pauli matrices `{X, Y, Z}` and `i` is the index of the qubit on which the Pauli acts.

### Check the code distance

We've defined the code. Now we'd like to make sure it's _correct_. In particular, it should have distance two.  QDK provides a `CodeProfile` that, among other things, can compute the code distance.

In [39]:
import qdk.ec as ec

code = ec.CodeProfile(c4)
code_distance = code.distance()
print("Code distance:", code_distance)

error = code_distance.witness.product
syndrome = code.syndrome_of(error)
effect = code.logical_effect_of(error)

print(f"Witness: {error}, (syndrome: {list(syndrome)}, logical effect: {effect})")
assert code_distance == error.weight == 2
assert code.is_logical(error)

Code distance: 2
Witness: XX, (syndrome: [], logical effect: X)


## 2. Build a baseline qodec

We have a code. But a code alone is a mathematical object: it doesn't describe how to _use_ it on a quantum computer. For that, we need a qodec, a formal description of an executable protocol.

Authoring a full qodec, including its instruction sets and gadgets, can be a substantial task. To get started, QDK provides `build_qodec`, which builds a baseline protocol from a code definition. There are many ways to do this; the `strategy` parameter selects the recipe.

We will use `bare-css/v1`, a strategy that measures syndromes with ancillas but adds no flag qubits.

The result has two layers: logical C4 operations and physical Stim operations. Each logical instruction has a _gadget_: a circuit that implements it using the next layer. Its encodings identify the physical qubits for each logical block; its equations describe checks and readouts.

Let's build the qodec and inspect the start of its YAML representation. Then we'll use `gadgets` to edit the protocol directly.

In [40]:
protocol = ec.build_qodec(c4, strategy="bare-css/v1", strict=False)
print(protocol.dumps()[:500], "...")

---
qodec.yaml:
  name: C4
  description: 'Built from the ''C4'' stabilizer code ([[4, 2]]). Strategy: bare-css/v1.'
  layers:
  - instruction_set: C4.isa.yaml
    codes:
      C4: C4.code.yaml
    gadgets:
      cx_all: cx_all.gadget.yaml
      measure_x_all: measure_x_all.gadget.yaml
      measure_z_all: measure_z_all.gadget.yaml
      prepare_x_all: prepare_x_all.gadget.yaml
      prepare_z_all: prepare_z_all.gadget.yaml
      syndrome: syndrome.gadget.yaml
  - instruction_set: stim.isa.yaml
 ...


### Inspect `prepare` instructions and gadgets

Let's look at `prepare_z_all` and its implementation together. The instruction produces one C4 block in logical $|00\rangle$. It has no input block and one output block, which encodes two logical qubits. The `stabilize` action requires both logical Z operators, `Z_0` and `Z_1`, to have eigenvalue +1.

In [41]:
instruction_set = protocol.layers[0].instruction_set
instruction_set.instructions["prepare_z_all"]

mnemonic: prepare_z_all
description: Prepare all 2 logical qubit(s) in |0⟩.
out: [C4]
action:
- stabilize: [Z_0, Z_1]


The instruction specifies the contract but does _not_ supply the implementation. That is the gadget's responsibility.  The corresponding `prepare_z_all` gadget is shown below.
It includes a circuit and parity checks. A _check_ is an equation that should evaluate to zero without faults. The equations include measurement bits and input/output encoding signs.

In [42]:
gadgets = protocol.layers[0].gadgets
gadgets["prepare_z_all"]

circuit:
  source: |
    R 0 1 2 3
    R 4
    H 4
    CX 4 0
    CX 4 1
    CX 4 2
    CX 4 3
    H 4
    R 5
    H 5
    CZ 5 0
    CZ 5 1
    CZ 5 2
    CZ 5 3
    H 5
    M 4 5
  format: stim
  out:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
out:
- C4: [0, 1, 2, 3]
checks:
- ['circuit.readouts[1]']
- ['circuit.readouts[0]', 'out[0].stabilizers[0]']
- ['out[0].stabilizers[1]']


The circuit source is Stim (qodec supports any circuit representation with an associated parser). So we can use `stim` directly to look at the circuit.

In [43]:
import stim
stim.Circuit(gadgets["prepare_z_all"].circuit.source).diagram("timeline-svg-html")

<svg viewBox="0 0 992 480"  version="1.1" xmlns="http://www.w3.org/2000/svg">
<g id="qubit_lines">
<path id="qubit_line:0" d="M64,64 L960,64 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="64">q0</text>
<path id="qubit_line:1" d="M64,128 L960,128 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="128">q1</text>
<path id="qubit_line:2" d="M64,192 L960,192 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="192">q2</text>
<path id="qubit_line:3" d="M64,256 L960,256 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="256">q3</text>
<path id="qubit_line:4" d="M64,320 L960,320 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="320">q4</text>
<path id="qubit_line:5" d="M64,384 L960,384 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="384">q5</text>
</g>
<rect x="80" y="48" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="64" fill="white">R</text>
<rect x="80" y="112" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="128" fill="white">R</text>
<rect x="80" y="176" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="192" fill="white">R</text>
<rect x="80" y="240" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="256" fill="white">R</text>
<rect x="80" y="304" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="320" fill="white">R</text>
<rect x="144" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="160" y="320">H</text>
<path d="M224,64 L224,320 " stroke="black"/>
<circle cx="224" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="224" cy="64" r="12" stroke="black" fill="white"/>
<path d="M212,64 L236,64 M224,52 L224,76 " stroke="black"/>
<path d="M288,128 L288,320 " stroke="black"/>
<circle cx="288" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="288" cy="128" r="12" stroke="black" fill="white"/>
<path d="M276,128 L300,128 M288,116 L288,140 " stroke="black"/>
<path d="M352,192 L352,320 " stroke="black"/>
<circle cx="352" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="352" cy="192" r="12" stroke="black" fill="white"/>
<path d="M340,192 L364,192 M352,180 L352,204 " stroke="black"/>
<path d="M416,256 L416,320 " stroke="black"/>
<circle cx="416" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="416" cy="256" r="12" stroke="black" fill="white"/>
<path d="M404,256 L428,256 M416,244 L416,268 " stroke="black"/>
<rect x="464" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="480" y="320">H</text>
<rect x="464" y="368" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="480" y="384" fill="white">R</text>
<rect x="528" y="368" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="544" y="384">H</text>
<path d="M608,64 L608,384 " stroke="black"/>
<circle cx="608" cy="384" r="12" s

Does this circuit prepare the state that the instruction claims it does? As with `CodeProfile`, we can use a `GadgetProfile` to analyze it. Its `objective` is the declared instruction; its `action` is the behavior computed from the circuit. Let's inspect the computed action and check that it matches the instruction.

In [44]:
from IPython.display import Markdown, display


def display_gadget_action(profile: ec.GadgetProfile) -> None:
    objective, action = profile.objective, profile.action
    if objective is None:
        result = "**Comparison unavailable:** no declared operation."
    elif action.is_equivalent_to(objective):
        result = "**Meets objective:** ✓"
    else:
        result = f"**Meets objective:** ✗\n\n{objective.why_not_equivalent_to(action)}"
    display(Markdown(f"**Computed action**\n\n```text\n{action}\n```\n\n{result}"))


preparation_profile = ec.GadgetProfile(gadgets["prepare_z_all"])
display_gadget_action(preparation_profile)

**Computed action**

```text
stabilizers:
  Z_0 = +1
  Z_1 = +1
```

**Meets objective:** ✓

### Inspect `cx`

Now let's look at `cx_all`. It takes two C4 blocks and applies CNOT between corresponding logical qubits. The `action` treats qubit indices as flat. The first C4 block corresponds to indices 0 and 1. The second C4 block corresponds to indices 2 and 3.

In [45]:
instruction_set.instructions["cx_all"]

mnemonic: cx_all
description: CNOT from the first block to the second, pairing logical qubits.
in: [C4, C4]
out: [C4, C4]
action:
- clifford:
    X_0: X_0 X_2
    X_1: X_1 X_3
    Z_2: Z_0 Z_2
    Z_3: Z_1 Z_3


Now, let's inspect the corresponding CNOT gadget. 

In [46]:
gadgets["cx_all"]

circuit:
  source: |
    CX 0 4
    CX 1 5
    CX 2 6
    CX 3 7
  format: stim
  in:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
    '4': qubit
    '5': qubit
    '6': qubit
    '7': qubit
  out:
    '0': qubit
    '1': qubit
    '2': qubit
    '3': qubit
    '4': qubit
    '5': qubit
    '6': qubit
    '7': qubit
in:
- C4: [0, 1, 2, 3]
- C4: [4, 5, 6, 7]
out:
- C4: [0, 1, 2, 3]
- C4: [4, 5, 6, 7]
checks:
- ['out[0].stabilizers[0]', 'in[0].stabilizers[0]', 'in[1].stabilizers[0]']
- ['out[0].stabilizers[1]', 'in[0].stabilizers[1]']
- ['out[1].stabilizers[0]', 'in[1].stabilizers[0]']
- ['out[1].stabilizers[1]', 'in[0].stabilizers[1]', 'in[1].stabilizers[1]']


The circuit is very simple: just transversal CNOTs across the two 4-qubit physical blocks. The `in/out` fields specify how to connect circuit labels to the logical blocks.
Note that, even though this gadget has no readouts, it _does_ include parity checks. The parity checks here relate the input stabilizers to the output stabilizers.

Next, we can look at the action of the gadget and verify that it implements the intended instruction.

In [47]:
cnot_profile = ec.GadgetProfile(gadgets["cx_all"])
display_gadget_action(cnot_profile)

**Computed action**

```text
mapping:
  X_0 → X_0 X_2
  Z_0 → Z_0
  X_1 → X_1 X_3
  Z_1 → Z_1
  X_2 → X_2
  Z_2 → Z_0 Z_2
  X_3 → X_3
  Z_3 → Z_1 Z_3
```

**Meets objective:** ✓

## 3. Check the entire qodec for correctness

We've inspected two instructions and their gadgets. Now we'd like to check the whole protocol. QDK provides `audit` to compare the circuits with their declared operations and equations. The audit should report no errors or warnings for the generated protocol. 

To see what the audit can tell us, we will deliberately remove some equations. The circuits will stay the same. Then we'll use QDK to derive the missing equations and check the protocol again.

In [48]:
report = ec.audit(protocol)
print(report)

audit: ok


### Remove the measurement readouts

The `measure_x_all` gadget should report two logical measurement results. Its readout equations tell us which physical measurement bits and frame signs to combine for each result.

Let's clear those equations. The circuit still measures the qubits, but the protocol no longer defines its logical results. The audit should report two `gadget/missing-observable` errors and show where the equations are missing.

In [49]:
gadgets["measure_x_all"].readouts.clear()
print(ec.audit(protocol))

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x_all'] (C4 -> stim)
readouts[0] has no equation for the required logical X_0 measurement
    Verified readout equation: ["in[0].x[0]", "circuit.readouts[0]", "circuit.readouts[1]"]

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x_all'] (C4 -> stim)
readouts[1] has no equation for the required logical X_1 measurement
    Verified readout equation: ["in[0].x[1]", "circuit.readouts[0]", "circuit.readouts[2]"]

audit: 2 error(s), 0 warning(s), 0 informational


### Remove the CNOT checks

Now let's remove the four checks from `cx_all`. These relate the input and output stabilizer signs. The circuit still performs the logical CNOT, but we have removed the equations that specify its output stabilizer signs. The audit should add four `gadget/incomplete-output-frame` warnings to the two readout errors.

In [50]:
gadgets["cx_all"].checks.clear()
print(ec.audit(protocol))

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x_all'] (C4 -> stim)
readouts[0] has no equation for the required logical X_0 measurement
    Verified readout equation: ["in[0].x[0]", "circuit.readouts[0]", "circuit.readouts[1]"]

[ERROR] gadget/missing-observable
layers[0].gadgets['measure_x_all'] (C4 -> stim)
readouts[1] has no equation for the required logical X_1 measurement
    Verified readout equation: ["in[0].x[1]", "circuit.readouts[0]", "circuit.readouts[2]"]

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['cx_all'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[0] (X_0 X_1 X_2 X_3)
    Code: C4; circuit support: ['0', '1', '2', '3'].
    Verified relation: ["out[0].stabilizers[0]", "in[0].stabilizers[0]", "in[1].stabilizers[0]"]

[WARNING] gadget/incomplete-output-frame
layers[0].gadgets['cx_all'] (C4 -> stim)
Declared relations do not determine out[0].stabilizers[1] (Z_0 Z_1 Z_2 Z_3)
    Code: C4; circuit support: ['0', '1', '2'

Replacing the missing gadget information can be tedious.  For situations like these, QDK offers `filled`.  This function accepts a partial qodec (or gadget) and attempts to fill it in based on the existing contents.

In [51]:
protocol = ec.filled(protocol)
gadgets = protocol.layers[0].gadgets
print(ec.audit(protocol))

audit: ok


## 4. Check the gadgets for fault tolerance

The qodec passes the audit. But the audit doesn't consider fault tolerance. What happens when a circuit has faults?

We checked the code distance in section 1. Now we will use `GadgetProfile.distance()` to check the _gadget distance_: the fewest allowed circuit faults that change the logical action, stay within the output codespaces, and leave every check and flag zero.

The default model is circuit noise: a Pauli error after a call, a flip of that call's recorded readout bits, or both.

Let's compute the distance of each gadget. Each result includes a _witness_: a set of faults that achieves the reported distance.

In [52]:
def report_gadget_distances(protocol: qodec.Qodec) -> None:
    distances = {
        mnemonic: ec.GadgetProfile(gadget).distance()
        for mnemonic, gadget in protocol.layers[0].gadgets.items()
    }
    rows = [
        "| Gadget | Distance | Witness |",
        "| --- | ---: | --- |",
    ]
    rows.extend(
        f"| `{mnemonic}` | {distance} | {distance.witness} |"
        for mnemonic, distance in sorted(distances.items())
    )
    display(Markdown("\n".join(rows)))

report_gadget_distances(protocol)

| Gadget | Distance | Witness |
| --- | ---: | --- |
| `cx_all` | 2 | X_4 after call 0; X_5 after call 1 |
| `measure_x_all` | 2 | X_0 after call 0; X_1 after call 1 |
| `measure_z_all` | 2 | flip call 0 readout 0; flip call 1 readout 0 |
| `prepare_x_all` | 1 | X_5 after call 18 |
| `prepare_z_all` | 1 | X_4 after call 7 |
| `syndrome` | 1 | X_4 after call 3 |

We're using a distance-two code, so we might hope that our gadgets also achieve distance two. Some of them do, but the `prepare` and `syndrome` gadgets do not. We fix that in the next section.

## 5. Increase the gadget distances

### Why the syndrome gadget has distance one

Let's inspect the `syndrome` circuit and the distance-one witness.


In [53]:
from IPython.display import HTML, display

circuit = gadgets["syndrome"].circuit
diagram = str(stim.Circuit(circuit.source).diagram("timeline-svg-html"))
call_text = "\n".join(
    f"{index}: {call.mnemonic} {call.operands}"
    for index, call in enumerate(circuit.calls())
)

display(HTML(
    '<div style="display:flex;gap:2rem;flex-wrap:wrap">'
    f'<pre>{diagram}</pre>'
    f'<pre>{call_text}</pre>'
    '</div>'
))


Ancilla 4 measures the X stabilizer with CNOTs. Ancilla 5 then measures the Z stabilizer with controlled-Z gates. We will follow faults on these ancillas through the remaining gates and inspect which checks and output signs they change.

In [54]:
idle_profile = ec.GadgetProfile(gadgets["syndrome"])
calls = gadgets["syndrome"].circuit.calls()
distance = idle_profile.distance()
location = distance.witness.product.locations[0]
call = calls[location.after_call]
effect = idle_profile.effects_of([distance.witness.product])[0]
print(f"{distance.witness} ({call.mnemonic} {call.operands}) → {effect}")

X_4 after call 3 (CX [4, 1]) → ['out[0].z[0]']


The offending fault occurs on the third call of the gadget, `CX 4 1`. It spreads through subsequent `CX` gates, ultimately flipping the output `z[0]` observable, but _not_ flipping the Z-stabilizer measurement. It is a logical `X_0` error.

But that's not the only problematic fault. The distance has several witnesses, which we examine below.

In [55]:
faults = [witness.product for witness in distance.witnesses]
effects = idle_profile.effects_of(faults)

for fault, effect in zip(faults, effects):
    location = fault.locations[0]
    call = calls[location.after_call]
    print(f"{fault}\t({call.mnemonic} {call.operands})\t→ {effect}")

X_4 after call 3	(CX [4, 1])	→ ['out[0].z[0]']
X_2 X_4 after call 4	(CX [4, 2])	→ ['out[0].z[0]']
X_5 after call 10	(CZ [5, 1])	→ ['out[0].x[1]']
Z_2 X_5 after call 11	(CZ [5, 2])	→ ['out[0].x[1]']


### Improve the syndrome circuit

All of the problematic faults are errors that occur on an ancilla qubit and then spread to multiple data qubits and go undetected. We need a circuit that avoids this phenomenon.
In this case, a circuit by [Reichardt](https://arxiv.org/pdf/1804.06995#page=4) does the trick. It interleaves the ancilla interactions so that any dangerous error spreads to the other ancilla's measurement result. It uses eight CNOTs and the same two ancillas, without adding a flag qubit.

Let's replace our circuit with Reichardt's. Then we'll use `filled` to recompute the parity checks and evaluate the new gadget distance.

In [56]:
reichardt_source = """
R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""
display(stim.Circuit(reichardt_source).diagram("timeline-svg-html"))

gadgets["syndrome"].circuit.source = reichardt_source
gadgets["syndrome"] = ec.filled(gadgets["syndrome"])

improved_syndrome_profile = ec.GadgetProfile(gadgets["syndrome"])
print("Improved syndrome distance:", improved_syndrome_profile.distance())
print(ec.audit(protocol))

<svg viewBox="0 0 864 480"  version="1.1" xmlns="http://www.w3.org/2000/svg">
<g id="qubit_lines">
<path id="qubit_line:0" d="M64,64 L832,64 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="64">q0</text>
<path id="qubit_line:1" d="M64,128 L832,128 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="128">q1</text>
<path id="qubit_line:2" d="M64,192 L832,192 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="192">q2</text>
<path id="qubit_line:3" d="M64,256 L832,256 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="256">q3</text>
<path id="qubit_line:4" d="M64,320 L832,320 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="320">q4</text>
<path id="qubit_line:5" d="M64,384 L832,384 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="384">q5</text>
</g>
<rect x="80" y="304" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="320" fill="white">R</text>
<rect x="80" y="368" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="384" fill="white">R</text>
<rect x="144" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="160" y="320">H</text>
<path d="M224,64 L224,320 " stroke="black"/>
<circle cx="224" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="224" cy="64" r="12" stroke="black" fill="white"/>
<path d="M212,64 L236,64 M224,52 L224,76 " stroke="black"/>
<path d="M288,192 L288,384 " stroke="black"/>
<circle cx="288" cy="192" r="12" stroke="none" fill="black"/>
<circle cx="288" cy="384" r="12" stroke="black" fill="white"/>
<path d="M276,384 L300,384 M288,372 L288,396 " stroke="black"/>
<path d="M352,64 L352,384 " stroke="black"/>
<circle cx="352" cy="64" r="12" stroke="none" fill="black"/>
<circle cx="352" cy="384" r="12" stroke="black" fill="white"/>
<path d="M340,384 L364,384 M352,372 L352,396 " stroke="black"/>
<path d="M416,128 L416,384 " stroke="black"/>
<circle cx="416" cy="128" r="12" stroke="none" fill="black"/>
<circle cx="416" cy="384" r="12" stroke="black" fill="white"/>
<path d="M404,384 L428,384 M416,372 L416,396 " stroke="black"/>
<path d="M480,192 L480,320 " stroke="black"/>
<circle cx="480" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="480" cy="192" r="12" stroke="black" fill="white"/>
<path d="M468,192 L492,192 M480,180 L480,204 " stroke="black"/>
<path d="M544,256 L544,320 " stroke="black"/>
<circle cx="544" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="544" cy="256" r="12" stroke="black" fill="white"/>
<path d="M532,256 L556,256 M544,244 L544,268 " stroke="black"/>
<path d="M608,128 L608,320 " stroke="black"/>
<circle cx="608" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="608" cy="128" r="12" stroke="black" fill="white"/>
<path d="M596,128 L620,128 M608,116 L608,140 " stroke="black"/>
<path d="M672,256 L672,384 " stroke="black"/>
<circle cx="672" cy="256" r="12" stroke="none" fill="black"/>
<circle cx="672" cy="384" r="12" stroke="black" fill="white"/>
<path d="M660,384 L684,384 M672,372 L672,396 " stroke="black"/>
<rect x="720" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="736" y="320">H</text>
<rect x="784" y="304" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" f

Improved syndrome distance: 2
audit: ok


### Detect analogous hook faults

The reordered circuit still lets an ancilla fault spread to two data qubits, but the other ancilla now detects it. The gadget distance of two is proof.

But it's possible to see this improvement explicitly by looking at how previously dangerous ancilla faults impact the new circuit.

In [57]:
improved_calls = gadgets["syndrome"].circuit.calls()
faults = [
    ec.FaultEvent.after(5, ec.Pauli("Z_5")),  
    ec.FaultEvent.after(7, ec.Pauli("X_4"))
]
effects = improved_syndrome_profile.effects_of(faults)

for fault, effect in zip(faults, effects):
    call = improved_calls[fault.locations[0].after_call]
    print(f"{fault} ({call.mnemonic} {call.operands}) →")
    for reference in effect.checks + effect.readouts:
        equation = gadgets["syndrome"].resolve(reference).value(tuple)
        print(f"  {reference} = {[str(term) for term in equation]}")
    for reference in effect.frames:
        print(f"  {reference}")
    print()

Z_5 after call 5 (CX [0, 5]) →
  checks[0] = ['circuit.readouts[0]', 'in[0].stabilizers[0]']
  checks[2] = ['circuit.readouts[0]', 'out[0].stabilizers[0]']
  out[0].x[0]

X_4 after call 7 (CX [4, 2]) →
  checks[1] = ['circuit.readouts[1]', 'in[0].stabilizers[1]']
  checks[3] = ['circuit.readouts[1]', 'out[0].stabilizers[1]']
  out[0].z[1]



The faults still flip logical operators (`Z_5` flips `out[0].x[0]`, `X_4` flips `out[0].z[1]`). But now, those faults are detected by parity checks, thereby increasing the distance.

### Improve both preparations with a flag

The `syndrome` gadget now looks good. We also need to improve the `prepare` gadgets. Let's look at the $|00\rangle$ circuit.

In [58]:
stim.Circuit(gadgets["prepare_z_all"].circuit.source).diagram("timeline-svg-html")

<svg viewBox="0 0 992 480"  version="1.1" xmlns="http://www.w3.org/2000/svg">
<g id="qubit_lines">
<path id="qubit_line:0" d="M64,64 L960,64 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="64">q0</text>
<path id="qubit_line:1" d="M64,128 L960,128 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="128">q1</text>
<path id="qubit_line:2" d="M64,192 L960,192 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="192">q2</text>
<path id="qubit_line:3" d="M64,256 L960,256 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="256">q3</text>
<path id="qubit_line:4" d="M64,320 L960,320 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="320">q4</text>
<path id="qubit_line:5" d="M64,384 L960,384 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="384">q5</text>
</g>
<rect x="80" y="48" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="64" fill="white">R</text>
<rect x="80" y="112" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="128" fill="white">R</text>
<rect x="80" y="176" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="192" fill="white">R</text>
<rect x="80" y="240" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="256" fill="white">R</text>
<rect x="80" y="304" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="320" fill="white">R</text>
<rect x="144" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="160" y="320">H</text>
<path d="M224,64 L224,320 " stroke="black"/>
<circle cx="224" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="224" cy="64" r="12" stroke="black" fill="white"/>
<path d="M212,64 L236,64 M224,52 L224,76 " stroke="black"/>
<path d="M288,128 L288,320 " stroke="black"/>
<circle cx="288" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="288" cy="128" r="12" stroke="black" fill="white"/>
<path d="M276,128 L300,128 M288,116 L288,140 " stroke="black"/>
<path d="M352,192 L352,320 " stroke="black"/>
<circle cx="352" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="352" cy="192" r="12" stroke="black" fill="white"/>
<path d="M340,192 L364,192 M352,180 L352,204 " stroke="black"/>
<path d="M416,256 L416,320 " stroke="black"/>
<circle cx="416" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="416" cy="256" r="12" stroke="black" fill="white"/>
<path d="M404,256 L428,256 M416,244 L416,268 " stroke="black"/>
<rect x="464" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="480" y="320">H</text>
<rect x="464" y="368" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="480" y="384" fill="white">R</text>
<rect x="528" y="368" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="544" y="384">H</text>
<path d="M608,64 L608,384 " stroke="black"/>
<circle cx="608" cy="384" r="12" s

This circuit is nearly identical to our original `syndrome` circuit, and so it's subject to the same problems: a fault on an ancilla spreads to an undetected logical error.

We'll fix this by instead using a unitary encoding circuit and a _flag_. A flag is a special parity check that gets promoted to a readout. It can then be used to reject noisy states, a process known as pre-selection.
Below, we add a `reject` flag to each gadget and bind it to the measurement. The `GadgetProfile.distance` calculation accounts for the flag automatically.

The $|++\rangle$ preparation is identical to the $|00\rangle$ preparation followed by transversal Hadamard gates.


In [59]:
flagged_preparation_source = """R 0 1 2 3 4
H 0
CX 0 4
CX 0 1
CX 0 2
CX 0 3
CX 0 4
"""
preparation_suffixes = {
    "prepare_z_all": "M 4\n",
    "prepare_x_all": "H 0 1 2 3\nM 4\n",
}
for mnemonic, suffix in preparation_suffixes.items():
    gadget = gadgets[mnemonic]
    gadget.implements.flags = ["reject"]
    gadget.circuit.source = flagged_preparation_source + suffix
    gadget.readouts = [{"reject": ["circuit.readouts[0]"]}]
    gadgets[mnemonic] = ec.filled(gadget)
    print(mnemonic, "distance:", ec.GadgetProfile(gadgets[mnemonic]).distance())
print(ec.audit(protocol))

prepare_z_all distance: 2
prepare_x_all distance: 2
audit: ok


In [60]:
stim.Circuit(gadgets["prepare_z_all"].circuit.source).diagram("timeline-svg-html")

<svg viewBox="0 0 608 416"  version="1.1" xmlns="http://www.w3.org/2000/svg">
<g id="qubit_lines">
<path id="qubit_line:0" d="M64,64 L576,64 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="64">q0</text>
<path id="qubit_line:1" d="M64,128 L576,128 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="128">q1</text>
<path id="qubit_line:2" d="M64,192 L576,192 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="192">q2</text>
<path id="qubit_line:3" d="M64,256 L576,256 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="256">q3</text>
<path id="qubit_line:4" d="M64,320 L576,320 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="320">q4</text>
</g>
<rect x="80" y="48" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="64" fill="white">R</text>
<rect x="80" y="112" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="128" fill="white">R</text>
<rect x="80" y="176" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="192" fill="white">R</text>
<rect x="80" y="240" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="256" fill="white">R</text>
<rect x="80" y="304" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="320" fill="white">R</text>
<rect x="144" y="48" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="160" y="64">H</text>
<path d="M224,64 L224,320 " stroke="black"/>
<circle cx="224" cy="64" r="12" stroke="none" fill="black"/>
<circle cx="224" cy="320" r="12" stroke="black" fill="white"/>
<path d="M212,320 L236,320 M224,308 L224,332 " stroke="black"/>
<path d="M288,64 L288,128 " stroke="black"/>
<circle cx="288" cy="64" r="12" stroke="none" fill="black"/>
<circle cx="288" cy="128" r="12" stroke="black" fill="white"/>
<path d="M276,128 L300,128 M288,116 L288,140 " stroke="black"/>
<path d="M352,64 L352,192 " stroke="black"/>
<circle cx="352" cy="64" r="12" stroke="none" fill="black"/>
<circle cx="352" cy="192" r="12" stroke="black" fill="white"/>
<path d="M340,192 L364,192 M352,180 L352,204 " stroke="black"/>
<path d="M416,64 L416,256 " stroke="black"/>
<circle cx="416" cy="64" r="12" stroke="none" fill="black"/>
<circle cx="416" cy="256" r="12" stroke="black" fill="white"/>
<path d="M404,256 L428,256 M416,244 L416,268 " stroke="black"/>
<path d="M480,64 L480,320 " stroke="black"/>
<circle cx="480" cy="64" r="12" stroke="none" fill="black"/>
<circle cx="480" cy="320" r="12" stroke="black" fill="white"/>
<path d="M468,320 L492,320 M480,308 L480,332 " stroke="black"/>
<rect x="528" y="304" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="544" y="320" fill="white">M</text>
<text text-anchor="middle" font-family="monospace" font-size="8" x="544" y="300">rec[0]</text>
</svg>

## 6. Inspect and save the improved qodec

We've changed the syndrome and preparation circuits. Now let's check the whole qodec again. The audit should report no errors or warnings, and all six gadgets should have distance two.

In [61]:
protocol.description = (
    "C4 built with bare-css/v1, with self-checking syndrome extraction "
    "and single-flag preparation circuits."
)
print(ec.audit(protocol))
report_gadget_distances(protocol);

audit: ok


| Gadget | Distance | Witness |
| --- | ---: | --- |
| `cx_all` | 2 | X_4 after call 0; X_5 after call 1 |
| `measure_x_all` | 2 | X_0 after call 0; X_1 after call 1 |
| `measure_z_all` | 2 | flip call 0 readout 0; flip call 1 readout 0 |
| `prepare_x_all` | 2 | X_1 after call 1; X_2 after call 2 |
| `prepare_z_all` | 2 | X_1 after call 1; X_2 after call 2 |
| `syndrome` | 2 | X_4 after call 0; Z_5 after call 5 |

### Save the qodec

We now have a qodec we'd like to keep. Use `save` to write its circuits and declarations into one YAML bundle.

In [62]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    saved_path = protocol.save(directory, single_file=True)
    assert qodec.Qodec.load(saved_path) == protocol
print("Round trip: OK")

Round trip: OK


## 7. Evaluate performance

First check that `qdk.simulation.run_qir` executes the improved protocol from section 6. `run_qir` runs each QIR call as the qodec instruction with exactly the callee's name, so the Q# program declares `prepare_z_all` and `measure_z_all` as intrinsic operations and calls them on one C4 block. It prepares logical $|00\rangle$ and measures both logical qubits in the Z basis. Every returned result should be `(Zero, Zero)` in this noiseless check.

This initial smoke test does not return the preparation flag; the runtime rejects a shot whose flag is raised. The benchmarks below instead use explicit C4 block instructions, returning both logical measurement outcomes per block and any preparation flags. They pass the same unmodified `protocol` to `run_qir`.

In [78]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binomtest
from qdk import Result, TargetProfile, qsharp
from qdk.simulation import NoiseConfig, run_qir
from qodec.instructions import InstructionCall

# Q#'s Base target rejects calls to custom measurements, though Base-profile QIR allows them.
qsharp.init(target_profile=TargetProfile.Adaptive_RI)
qsharp.eval("""
operation prepare_z_all(block : Qubit) : Unit { body intrinsic; }

@Measurement()
operation measure_z_all(block : Qubit) : (Result, Result) { body intrinsic; }
""")
qir_program = qsharp.compile(
    "{ use block = Qubit(); prepare_z_all(block); measure_z_all(block) }"
)

### Check the noiseless result

Every shot should return `(Zero, Zero)`. Use `on_shot_failure="raise"` here so a rejected or undecodable shot fails the check rather than disappearing from the results.

In [73]:
noiseless_shots = 20
noiseless_results = run_qir(
    qir_program,
    shots=noiseless_shots,
    seed=7,
    type="clifford",
    qodec=protocol,
    on_shot_failure="raise",
)
assert noiseless_results == [(Result.Zero, Result.Zero)] * noiseless_shots
print(f"Noiseless check: {noiseless_shots} correct shots, none discarded.")

Noiseless check: 20 correct shots, none discarded.


### Build QIR experiments, not fault-effect tables

For example, `cx_all Z` calls `prepare_z_all` on blocks 0 and 1, `cx_all` on the pair of blocks, then `measure_z_all` on each block. `cx_all` acts on both corresponding logical pairs. Both blocks start in logical 00, so the expected four measurement bits are zero. `SPAM ZZ` is the same sequence without `cx_all`.

X tests call `prepare_x_all` and `measure_x_all` directly. Syndrome tests call `syndrome` directly. There are no logical H, X, single-pair CNOT, or movement substitutions. The call lists are checked against the actual C4 ISA, including operand counts and block lifetimes.

The notebook's QIR adapter calls each C4 instruction by its exact mnemonic, such as `@prepare_z_all`. Its arguments are opaque C4 block handles followed by result slots: logical readouts first, then declared flags. It records flags before logical measurements in the returned array. `run_qir` dispatches each call to the qodec instruction of the same name. A call that creates a block replaces the implicit preparation of that QIR qubit. If that interface is unsupported, the notebook leaves the exception visible; it does not lower the benchmark to a different instruction set.

The noise configuration uses total depolarizing probability $p$ after physical H, X, Y, Z, and CX gates, plus `mresetz.x = p` for reset/measurement state noise. In the current runtime, `mresetz` noise acts on the state after measurement; it does **not** flip the returned measurement bit. A complete circuit-noise model still needs physical readout-bit noise support. We do not simulate that missing channel outside `run_qir`.

Use `on_shot_failure="discard"` and count every requested attempt. Also discard returned shots with any raised preparation flag. Yield counts the remaining shots; an accepted shot is wrong if any logical measurement bit differs from the expected bit string. Flag values are returned by the runtime, not inferred from a fault model. No ideal boundary checks, retries, or confidence-based selection are added here.

In [79]:
def benchmark_qir(calls: list[InstructionCall]) -> tuple[str, int, int]:
    """Encode named C4 calls; integer operands are zero-based block IDs."""
    isa = protocol.layers[0].instruction_set
    body = []
    declarations = {}
    live_blocks: set[int] = set()
    used_blocks: set[int] = set()
    logical_results = []
    flag_results = []
    result_count = 0
    for call in calls:
        instruction = isa.instructions[call.mnemonic]
        if call.arguments or call.select or instruction.parameters:
            raise ValueError("These benchmarks use unparameterized calls and postselect returned flags.")
        blocks = []
        for operand in call.operands:
            if not isinstance(operand, int) or operand < 0:
                raise ValueError(f"Expected a nonnegative C4 block ID: {operand}")
            blocks.append(operand)
        arity = max(len(instruction.inputs), len(instruction.outputs))
        if len(blocks) != arity or len(set(blocks)) != arity:
            raise ValueError(f"Invalid operands for {call.mnemonic}: {blocks}")
        inputs = set(blocks[:len(instruction.inputs)])
        outputs = set(blocks[:len(instruction.outputs)])
        if not inputs <= live_blocks or (outputs - inputs) & live_blocks:
            raise ValueError(f"Invalid block lifetime at {call.mnemonic}: {blocks}")
        live_blocks = (live_blocks - inputs) | outputs
        used_blocks.update(blocks)
        outcome_count = instruction.observe_count
        readout_count = outcome_count + len(instruction.flags)
        readouts = list(range(result_count, result_count + readout_count))
        logical_results.extend(readouts[:outcome_count])
        flag_results.extend(readouts[outcome_count:])
        result_count += readout_count
        arguments = [f"%Qubit* inttoptr (i64 {block} to %Qubit*)" for block in blocks]
        arguments += [f"%Result* inttoptr (i64 {result} to %Result*)" for result in readouts]
        symbol = call.mnemonic
        body.append(f"  call void @{symbol}({', '.join(arguments)})")
        types = ["%Qubit*"] * arity + ["%Result*"] * readout_count
        declarations[symbol] = f"declare void @{symbol}({', '.join(types)})"
    if live_blocks or not logical_results:
        raise ValueError("A benchmark must finish by measuring every prepared block.")
    block_count = len(used_blocks)
    if used_blocks != set(range(block_count)):
        raise ValueError("C4 block IDs must be contiguous, starting at zero.")
    recorded = flag_results + logical_results
    body.append(f"  call void @__quantum__rt__array_record_output(i64 {len(recorded)}, i8* null)")
    body.extend(
        f"  call void @__quantum__rt__result_record_output(%Result* inttoptr (i64 {result} to %Result*), i8* null)"
        for result in recorded
    )
    body_text = "\n".join(body)
    declaration_text = "\n".join(declarations.values())
    program = f'''%Qubit = type opaque
%Result = type opaque

define void @ENTRYPOINT__main() #0 {{
entry:
{body_text}
  ret void
}}

{declaration_text}
declare void @__quantum__rt__array_record_output(i64, i8*)
declare void @__quantum__rt__result_record_output(%Result*, i8*)
attributes #0 = {{ "entry_point" "qir_profiles"="base_profile" "required_num_qubits"="{block_count}" "required_num_results"="{result_count}" }}
'''
    return program, len(logical_results), len(flag_results)


def runtime_noise(probability: float) -> NoiseConfig:
    noise = NoiseConfig()
    for gate in (noise.h, noise.x, noise.y, noise.z, noise.cx):
        gate.set_depolarizing(probability)
    noise.mresetz.x = probability
    return noise

### Define test circuits and matched SPAM references

Each Z/X test prepares all logical qubits in the positive Z/X eigenstate and measures in that basis. Each block contributes two logical bits. `SPAM Z` and `SPAM X` measure preparation and readout together; those two instruction error rates cannot be separated by this comparison. The `cx_all` tests use matched two-block references, and the syndrome tests use single-block references. These selected-input tests measure bit/phase errors; they are not full process tomography or a separate proof of the CNOT action, which is checked by the earlier gadget audit.

In [80]:
assert set(protocol.layers[0].instruction_set.instructions) == {
    "prepare_z_all", "prepare_x_all", "measure_z_all", "measure_x_all", "cx_all", "syndrome",
}
benchmark_calls = {}
for basis in ("z", "x"):
    label = basis.upper()
    single_block = [
        InstructionCall(f"prepare_{basis}_all", operands=[0]),
        InstructionCall(f"measure_{basis}_all", operands=[0]),
    ]
    two_blocks = [
        InstructionCall(f"{operation}_{basis}_all", operands=[block])
        for operation in ("prepare", "measure") for block in (0, 1)
    ]
    benchmark_calls[f"SPAM {label}"] = (single_block, f"SPAM {label}")
    benchmark_calls[f"SPAM {label}{label}"] = (two_blocks, f"SPAM {label}{label}")
    benchmark_calls[f"cx_all {label}"] = (
        two_blocks[:2] + [InstructionCall("cx_all", operands=[0, 1])] + two_blocks[2:],
        f"SPAM {label}{label}",
    )
    benchmark_calls[f"syndrome {label}"] = (
        single_block[:1] + [InstructionCall("syndrome", operands=[0])] + single_block[1:],
        f"SPAM {label}",
    )
# Each entry is (QIR, expected logical readouts, flag count, SPAM reference).
benchmarks = {}
for name, (calls, reference) in benchmark_calls.items():
    program, logical_readout_count, flag_count = benchmark_qir(calls)
    benchmarks[name] = (program, [Result.Zero] * logical_readout_count, flag_count, reference)
    print(name + ": " + "; ".join(f"{call.mnemonic} {call.operands}" for call in calls))

SPAM Z: prepare_z_all [0]; measure_z_all [0]
SPAM ZZ: prepare_z_all [0]; prepare_z_all [1]; measure_z_all [0]; measure_z_all [1]
cx_all Z: prepare_z_all [0]; prepare_z_all [1]; cx_all [0, 1]; measure_z_all [0]; measure_z_all [1]
syndrome Z: prepare_z_all [0]; syndrome [0]; measure_z_all [0]
SPAM X: prepare_x_all [0]; measure_x_all [0]
SPAM XX: prepare_x_all [0]; prepare_x_all [1]; measure_x_all [0]; measure_x_all [1]
cx_all X: prepare_x_all [0]; prepare_x_all [1]; cx_all [0, 1]; measure_x_all [0]; measure_x_all [1]
syndrome X: prepare_x_all [0]; syndrome [0]; measure_x_all [0]


### Sample error rate and yield

Use 100 attempts per program per noise level for fast runtime iterations. The table includes the signed difference from the matched SPAM error rate, in percentage points, and the ratio of yields. These are diagnostics, not isolated instruction probabilities: postselection changes the accepted populations, errors can cancel, and finite sampling can give a negative difference or a yield ratio above one. Keep the raw error rates and yields. Exceptions are not caught; an incomplete sweep must not produce a plot.

In [82]:
fault_probabilities = [1e-3, 3e-3, 1e-2, 3e-2, 5e-2]
shots_per_point = 100
sweep = []
sweep_complete = False
started = perf_counter()
for probability_index, probability in enumerate(fault_probabilities):
    for benchmark_index, (name, (program, expected, flag_count, reference)) in enumerate(benchmarks.items()):
        print(f"Running {name}, p={probability:g} ...", flush=True)
        print(program, flush=True)
        results = run_qir(
            program, qodec=protocol, noise=runtime_noise(probability),
            shots=shots_per_point,
            seed=42 + probability_index * len(benchmarks) + benchmark_index,
            type="clifford", on_shot_failure="discard",
        )
        if any(
            not isinstance(result, list) or len(result) != flag_count + len(expected)
            or any(bit not in (Result.Zero, Result.One) for bit in result)
            for result in results
        ):
            raise ValueError(f"Unexpected QIR result shape for {name}: {results[:3]}")
        accepted_results = [
            result[flag_count:] for result in results
            if all(flag == Result.Zero for flag in result[:flag_count])
        ]
        accepted_count = len(accepted_results)
        if accepted_count == 0:
            raise RuntimeError(f"No {name} shots accepted at p={probability:g}.")
        sweep.append({
            "benchmark": name,
            "reference": reference,
            "fault_probability": probability,
            "attempted": shots_per_point,
            "accepted": accepted_count,
            "wrong": sum(result != expected for result in accepted_results),
            "discarded": shots_per_point - accepted_count,
        })
sweep_complete = True
print(f"Executed {len(sweep) * shots_per_point:,} QIR attempts in {perf_counter() - started:.3f} s.")
by_case = {(point["benchmark"], point["fault_probability"]): point for point in sweep}
rows = [
    "| Benchmark | Fault probability | Attempted | Accepted | Wrong | Error minus SPAM (pp) | Yield / SPAM yield |",
    "| :--- | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for point in sweep:
    reference = by_case[point["reference"], point["fault_probability"]]
    excess_error = point["wrong"] / point["accepted"] - reference["wrong"] / reference["accepted"]
    relative_yield = (point["accepted"] / point["attempted"]) / (reference["accepted"] / reference["attempted"])
    rows.append(
        f"| {point['benchmark']} | {point['fault_probability']:.4g} | {point['attempted']} | "
        f"{point['accepted']} | {point['wrong']} | {100 * excess_error:+.2f} | {relative_yield:.3f} |"
    )
display(Markdown("\n".join(rows)))

Running SPAM Z, p=0.001 ...
%Qubit = type opaque
%Result = type opaque

define void @ENTRYPOINT__main() #0 {
entry:
  call void @__quantum__qis__prepare_z_all__body(%Qubit* inttoptr (i64 0 to %Qubit*), %Result* inttoptr (i64 0 to %Result*))
  call void @__quantum__qis__measure_z_all__body(%Qubit* inttoptr (i64 0 to %Qubit*), %Result* inttoptr (i64 1 to %Result*), %Result* inttoptr (i64 2 to %Result*))
  call void @__quantum__rt__array_record_output(i64 3, i8* null)
  call void @__quantum__rt__result_record_output(%Result* inttoptr (i64 0 to %Result*), i8* null)
  call void @__quantum__rt__result_record_output(%Result* inttoptr (i64 1 to %Result*), i8* null)
  call void @__quantum__rt__result_record_output(%Result* inttoptr (i64 2 to %Result*), i8* null)
  ret void
}

declare void @__quantum__qis__prepare_z_all__body(%Qubit*, %Result*)
declare void @__quantum__qis__measure_z_all__body(%Qubit*, %Result*, %Result*)
declare void @__quantum__rt__array_record_output(i64, i8*)
declare void @_

KeyError: 'prepare_z_all'

### Compare conditional logical error rate and yield

The left plot estimates $P(\text{wrong program result}\mid\text{accepted})$; the right estimates $P(\text{accepted})$. These are complete QIR sequences, including SPAM and decoding. Compare each test with its matched SPAM curve rather than treating raw subtraction as an isolated error probability. The dashed break-even line marks equal logical error and physical fault probabilities; the percentage y-axis puts this line at $100p$.

Error bars are pointwise 95% Wilson binomial confidence intervals. Both panels use log-log axes. Zero cannot be plotted on a log axis: open downward triangles mark the upper interval endpoint for zero-error samples, without connecting them as measured rates. At only 100 attempts, small error rates are poorly resolved; increase `shots_per_point` after the runtime paths work. The signed SPAM differences stay in the table, not on a logarithmic axis. The noiseless check is separate because $p=0$ cannot appear on a logarithmic x-axis.

In [ ]:
if not sweep_complete or len(sweep) != len(benchmarks) * len(fault_probabilities):
    raise RuntimeError("The QIR sweep did not finish; do not plot partial or stale results.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
metrics = [
    ("wrong", "accepted", "Conditional logical error rate", "Wrong / accepted (%)"),
    ("accepted", "attempted", "Yield", "Accepted / attempted (%)"),
]
markers = ["o", "s", "^", "v", "D", "P", "X", "*"]
for ax, (numerator, denominator, title, ylabel) in zip(axes, metrics):
    for name, marker in zip(benchmarks, markers):
        points = [point for point in sweep if point["benchmark"] == name]
        rates = np.array([point[numerator] / point[denominator] for point in points])
        intervals = np.array([
            binomtest(point[numerator], point[denominator]).proportion_ci(
                confidence_level=0.95, method="wilson"
            )
            for point in points
        ])
        errors = np.vstack((rates - intervals[:, 0], intervals[:, 1] - rates))
        probabilities = np.array([point["fault_probability"] for point in points])
        zero_errors = (rates == 0) & (numerator == "wrong")
        curve = ax.errorbar(
            probabilities, 100 * np.where(zero_errors, np.nan, rates),
            yerr=100 * np.where(zero_errors, np.nan, errors),
            fmt=marker + "-", capsize=3, label=name,
        )
        if zero_errors.any():
            ax.scatter(
                probabilities[zero_errors], 100 * intervals[zero_errors, 1],
                marker="v", facecolors="none", edgecolors=curve.lines[0].get_color(),
                zorder=3,
            )
    ax.set(xscale="log", yscale="log", xlabel="Physical fault probability per location", ylabel=ylabel, title=title)
    ax.grid(True, which="both", alpha=0.25)

(breakeven_line,) = axes[0].plot(
    fault_probabilities, 100 * np.array(fault_probabilities),
    "k--", linewidth=1, label="Break-even",
)
axes[0].legend(handles=[breakeven_line], loc="lower left", fontsize="small")
axes[0].text(
    0.03, 0.97, "Open triangles: 95% upper bounds",
    transform=axes[0].transAxes, va="top", fontsize="small",
)
axes[1].set_ylim(top=100)
axes[1].legend(loc="lower left", fontsize="small")
fig.suptitle("C4 via run_qir: program error and yield, including SPAM")
plt.show()

## What we established

We started with the C4 code and built a baseline qodec. We used the audit to find missing equations, then used gadget profiles to find faults and test better circuits. The resulting six gadgets pass the audit and each has distance two. We can also save and load the protocol without changing its declarations.

Section 7 defines benchmarks entirely in the C4 ISA, observing both logical slots of each measured block. It sends the named instruction calls through `run_qir` with the improved qodec and matched SPAM references. Runtime exceptions remain visible, and only completed runs support performance claims. The current noise configuration does not include physical readout-bit flips.

Complete execution still requires named QIR-to-ISA dispatch. Broader performance claims also need additional input states, physical readout noise, decoder-confidence selection, instruction-local retries, and realistic connectivity, timing, idle, and correlated faults.